# Tree Implementation Lab

```{contents}
:local:
:depth: 2
```

## An Ordered Service Desk

```{index} binary search tree; ordered set; service desk
```

A service desk assigns unique integer IDs to active tickets. Staff need to locate an ID, list active IDs in order, find the smallest active ID, and remove closed tickets. The BST is an index over IDs; it does not imply that the smallest ID is the most urgent ticket. A queue preserves arrival order, while this tree preserves key order.

The `TicketTree` class from the previous section supplies the basic operations. Each example below includes its own class and setup so you can run it independently. Download {download}`the complete source and verification program <../../demos/17/TicketTree.cs>` for use in a console project.

Before adding more operations, write down the contract: no duplicate IDs; keys strictly increase in inorder traversal; Count equals the number of reachable nodes; unsuccessful updates preserve membership. Empty minimum lookup reports false rather than inventing an ID.

In [ ]:
using System;
using System.Collections.Generic;

var tickets = new TicketTree();
foreach (int id in new[] { 40, 20, 60, 10, 30, 50, 70 }) tickets.Add(id);
Console.WriteLine("active=" + string.Join(", ", tickets.InOrder()));
Console.WriteLine($"find 30={tickets.Contains(30)}, find 35={tickets.Contains(35)}");
tickets.TryMinimum(out int first);
Console.WriteLine($"minimum={first}, count={tickets.Count}, height={tickets.Height}");

public class TicketTree
{
    private Node? root;
    public int Count { get; private set; }
    public int Height => NodeHeight(root);

    public bool Add(int id)
    {
        bool added = false;
        root = Insert(root, id, ref added);
        if (added) Count++;
        return added;
    }

    public bool Contains(int id)
    {
        Node? current = root;
        while (current is not null)
        {
            if (id == current.Id) return true;
            current = id < current.Id ? current.Left : current.Right;
        }
        return false;
    }

    public bool Remove(int id)
    {
        bool removed = false;
        root = Delete(root, id, ref removed);
        if (removed) Count--;
        return removed;
    }

    public bool TryMinimum(out int id)
    {
        if (root is null) { id = default; return false; }
        id = Minimum(root).Id;
        return true;
    }

    public List<int> InOrder()
    {
        var ids = new List<int>();
        Visit(root, ids);
        return ids;
    }

    public bool IsValid() => Validate(root, long.MinValue, long.MaxValue);

    private static Node Insert(Node? node, int id, ref bool added)
    {
        if (node is null) { added = true; return new Node(id); }
        if (id < node.Id) node.Left = Insert(node.Left, id, ref added);
        else if (id > node.Id) node.Right = Insert(node.Right, id, ref added);
        return node;
    }

    private static Node? Delete(Node? node, int id, ref bool removed)
    {
        if (node is null) return null;
        if (id < node.Id) node.Left = Delete(node.Left, id, ref removed);
        else if (id > node.Id) node.Right = Delete(node.Right, id, ref removed);
        else
        {
            removed = true;
            if (node.Left is null) return node.Right;
            if (node.Right is null) return node.Left;
            Node successor = Minimum(node.Right);
            node.Id = successor.Id;
            bool successorRemoved = false;
            node.Right = Delete(node.Right, successor.Id, ref successorRemoved);
        }
        return node;
    }

    private static Node Minimum(Node node)
    {
        while (node.Left is not null) node = node.Left;
        return node;
    }

    private static void Visit(Node? node, List<int> ids)
    {
        if (node is null) return;
        Visit(node.Left, ids);
        ids.Add(node.Id);
        Visit(node.Right, ids);
    }

    private static int NodeHeight(Node? node) => node is null ? -1
        : 1 + Math.Max(NodeHeight(node.Left), NodeHeight(node.Right));

    private static bool Validate(Node? node, long lower, long upper)
    {
        if (node is null) return true;
        return lower < node.Id && node.Id < upper
            && Validate(node.Left, lower, node.Id)
            && Validate(node.Right, node.Id, upper);
    }

    private class Node
    {
        public int Id { get; set; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int id) => Id = id;
    }
}

## Range Queries and Pruning

```{index} binary search tree; range query; pruning
```

Suppose staff want all active IDs from 25 through 60, inclusive. One method gets every ID and filters the resulting list. A BST can also skip subtrees that cannot contain a match. **Pruning** means using a proved constraint to avoid visiting part of a structure.

At key 20, the entire left subtree is below the lower bound 25. At key 60, the entire right subtree is above the upper bound 60. For keys within the interval, process left, node, and right to keep the result ordered. Bounds are inclusive for the query even though the BST's ordering invariant is strict.

The standalone example exposes nodes only to make the recursion visible. The same method could be private inside `TicketTree`, with a public `Range(lower, upper)` wrapper.

In [ ]:
using System;
using System.Collections.Generic;

var root = new Node(40)
{
    Left = new Node(20) { Left = new Node(10), Right = new Node(30) },
    Right = new Node(60) { Left = new Node(50), Right = new Node(70) }
};
var matches = new List<int>();
int visited = 0;
Range(root, 25, 60, matches);
Console.WriteLine("matches=" + string.Join(", ", matches));
Console.WriteLine($"nodes visited={visited}");

void Range(Node? node, int lower, int upper, List<int> result)
{
    if (node is null || lower > upper) return;
    visited++;
    if (lower < node.Id) Range(node.Left, lower, upper, result);
    if (lower <= node.Id && node.Id <= upper) result.Add(node.Id);
    if (node.Id < upper) Range(node.Right, lower, upper, result);
}

class Node
{
    public int Id { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }
    public Node(int id) => Id = id;
}

The result is `30, 40, 50, 60`. The method visits five of the seven nodes; it skips 10 and 70. The number of matching keys alone does not determine cost: searching for an empty interval can still walk a long path. In a tree of height h returning k keys, this traversal takes O(h + k + 1) time; a chain or a query covering the whole tree can require Θ(n). Its recursion uses O(h + 1) space, plus the returned values.

Pruning is safe because of the BST invariant. Applying this method to an arbitrary binary tree could silently omit valid matches. Test a range with no matches, one exact key, all keys, and reversed bounds. The method defines reversed bounds as an empty query.

## Level Order with a Queue

```{index} tree; level order traversal; queue
```

Depth-first traversals finish descendants before moving to another branch. **Level order** visits nodes by depth: root first, then its children, then their children. Use the Chapter 16 queue as a worklist of nodes discovered but not yet processed.

When you remove a node from the queue, append its existing children from left to right. The queue keeps earlier levels ahead of later ones. In a proper tree, each non-root node has exactly one parent, so it is enqueued once. Graphs with shared descendants or cycles need an additional visited set, a topic for Chapter 19.

In [ ]:
using System;
using System.Collections.Generic;

var root = new Node(40)
{
    Left = new Node(20) { Left = new Node(10), Right = new Node(30) },
    Right = new Node(60) { Left = new Node(50), Right = new Node(70) }
};
var pending = new Queue<Node>();
pending.Enqueue(root);
while (pending.Count > 0)
{
    Node node = pending.Dequeue();
    Console.WriteLine($"visit={node.Id}; pending before children={pending.Count}");
    if (node.Left is not null) pending.Enqueue(node.Left);
    if (node.Right is not null) pending.Enqueue(node.Right);
}

class Node
{
    public int Id { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }
    public Node(int id) => Id = id;
}

The pending IDs evolve as `40`, then `20, 60`, then `60, 10, 30`, then `10, 30, 50, 70`. Dequeue processes the front, and child discovery appends at the back. The visit order is `40, 20, 60, 10, 30, 50, 70`, which differs from preorder and inorder.

Each node is enqueued and dequeued once, giving Θ(n) work. Queue space depends on the tree's width and can reach O(n), even when height is small. A recursive depth-first traversal uses space proportional to height instead. For an empty tree, initialize no pending node and produce no output.

## Verify Updates Against a Model

```{index} binary search tree; testing; SortedSet
```

A test should check the operation's result and the structure left behind. Use `SortedSet<int>` as an independent reference for membership and sorted output. It implements the same unique-key contract through a different library implementation. Comparing only printed output from one hand-built tree would miss many reference-update errors.

After each update, compare the returned Boolean, Count, inorder keys, and ordering validator. Also check the minimum and membership. The example deliberately removes a root with two children while its successor has a right child, drains the tree, and reuses it.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var tree = new TicketTree();
var model = new SortedSet<int>();
void Check()
{
    if (!tree.IsValid() || tree.Count != model.Count
        || !tree.InOrder().SequenceEqual(model)) throw new Exception("Tree/model mismatch.");
    bool found = tree.TryMinimum(out int minimum);
    if (found != (model.Count > 0) || (found && minimum != model.Min))
        throw new Exception("Minimum mismatch.");
}
foreach (int id in new[] { 8, 3, 12, 1, 6, 10, 14, 11, 8, int.MinValue, int.MaxValue })
{
    if (tree.Add(id) != model.Add(id)) throw new Exception("Add result mismatch.");
    Check();
}
foreach (int id in new[] { 8, 99, 3, 12, 1, 6, 10, 14, 11, int.MinValue, int.MaxValue })
{
    if (tree.Remove(id) != model.Remove(id)) throw new Exception("Remove result mismatch.");
    Check();
}
if (tree.Remove(8) || tree.Contains(8)) throw new Exception("Empty boundary failed.");
tree.Add(0); model.Add(0); Check();
Console.WriteLine($"checks passed; reused minimum={model.Min}; count={tree.Count}");

public class TicketTree
{
    private Node? root;
    public int Count { get; private set; }
    public int Height => NodeHeight(root);

    public bool Add(int id)
    {
        bool added = false;
        root = Insert(root, id, ref added);
        if (added) Count++;
        return added;
    }

    public bool Contains(int id)
    {
        Node? current = root;
        while (current is not null)
        {
            if (id == current.Id) return true;
            current = id < current.Id ? current.Left : current.Right;
        }
        return false;
    }

    public bool Remove(int id)
    {
        bool removed = false;
        root = Delete(root, id, ref removed);
        if (removed) Count--;
        return removed;
    }

    public bool TryMinimum(out int id)
    {
        if (root is null) { id = default; return false; }
        id = Minimum(root).Id;
        return true;
    }

    public List<int> InOrder()
    {
        var ids = new List<int>();
        Visit(root, ids);
        return ids;
    }

    public bool IsValid() => Validate(root, long.MinValue, long.MaxValue);

    private static Node Insert(Node? node, int id, ref bool added)
    {
        if (node is null) { added = true; return new Node(id); }
        if (id < node.Id) node.Left = Insert(node.Left, id, ref added);
        else if (id > node.Id) node.Right = Insert(node.Right, id, ref added);
        return node;
    }

    private static Node? Delete(Node? node, int id, ref bool removed)
    {
        if (node is null) return null;
        if (id < node.Id) node.Left = Delete(node.Left, id, ref removed);
        else if (id > node.Id) node.Right = Delete(node.Right, id, ref removed);
        else
        {
            removed = true;
            if (node.Left is null) return node.Right;
            if (node.Right is null) return node.Left;
            Node successor = Minimum(node.Right);
            node.Id = successor.Id;
            bool successorRemoved = false;
            node.Right = Delete(node.Right, successor.Id, ref successorRemoved);
        }
        return node;
    }

    private static Node Minimum(Node node)
    {
        while (node.Left is not null) node = node.Left;
        return node;
    }

    private static void Visit(Node? node, List<int> ids)
    {
        if (node is null) return;
        Visit(node.Left, ids);
        ids.Add(node.Id);
        Visit(node.Right, ids);
    }

    private static int NodeHeight(Node? node) => node is null ? -1
        : 1 + Math.Max(NodeHeight(node.Left), NodeHeight(node.Right));

    private static bool Validate(Node? node, long lower, long upper)
    {
        if (node is null) return true;
        return lower < node.Id && node.Id < upper
            && Validate(node.Left, lower, node.Id)
            && Validate(node.Right, node.Id, upper);
    }

    private class Node
    {
        public int Id { get; set; }
        public Node? Left { get; set; }
        public Node? Right { get; set; }
        public Node(int id) => Id = id;
    }
}

The model comparison detects missing or duplicated keys, while `IsValid` checks ancestor bounds. Neither replaces an explanation of why deletion preserves the invariant. Tests sample executions; the successor argument explains the whole two-child case.

For broader checking, generate a reproducible sequence of adds, removals, and membership queries and compare after each step. Keep explicit boundary cases because a random generator may never produce the smallest and largest integers, a singleton, or a successor with a right child. Avoid timing tiny examples to infer Big-O behavior: count operations and reason about height.

## Exercise

```{index} binary search tree; subtree size; exercise
```

In [ ]:
// Exercise: Count active IDs inside an inclusive interval.
// Complete CountRange; skip subtrees proved outside the interval.
using System;
var root = new Node(40)
{
    Left = new Node(20) { Left = new Node(10), Right = new Node(30) },
    Right = new Node(60) { Left = new Node(50), Right = new Node(70) }
};
Console.WriteLine(CountRange(root, 25, 60));
Console.WriteLine(CountRange(root, 80, 90));
int CountRange(Node? node, int lower, int upper)
{
    // Your code starts here.
    /* TODO */
    // Your code ends here.
}
class Node
{
    public int Id { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }
    public Node(int id) => Id = id;
}

In [1]:
// Solution
using System;
var root = new Node(40)
{
    Left = new Node(20) { Left = new Node(10), Right = new Node(30) },
    Right = new Node(60) { Left = new Node(50), Right = new Node(70) }
};
Console.WriteLine(CountRange(root, 25, 60));
Console.WriteLine(CountRange(root, 80, 90));
int CountRange(Node? node, int lower, int upper)
{
    if (node is null || lower > upper) return 0;
    if (node.Id < lower) return CountRange(node.Right, lower, upper);
    if (node.Id > upper) return CountRange(node.Left, lower, upper);
    return 1 + CountRange(node.Left, lower, upper) + CountRange(node.Right, lower, upper);
}
class Node
{
    public int Id { get; }
    public Node? Left { get; set; }
    public Node? Right { get; set; }
    public Node(int id) => Id = id;
}

4
0


## Summary

```{index} tree operations; correctness
```

Use the BST ordering rule to select or prune subtrees. Preserve references when updates replace subtree roots. Match the collection's membership contract with a reference model, and state costs in terms of actual height. Level order connects trees to queue worklists; range queries connect ordered storage to reporting tasks.

```{rubric} Footnotes
```

[^1]: These examples index integer IDs only. A production ticket system would store records separately or keep keys and record values together, with a comparison policy that stays consistent across all operations.